In [16]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import SplineTransformer

In [30]:
df = pd.read_csv('../data/processed_training_data.csv', parse_dates=['Time'])
df = df.set_index('Time')
df.head()

,GHI(W/m2),Windspeed(m/s),Solar Energy(MWh),Wind Energy(MWh),Total Energy(MWh),Battery_Charge(MWh),Stored Energy(MWh)
Time,,,,,,,
2014-01-01 00:00:00,0.0,11.37,0.0,8.506240,8.506240,0.0,0.0
2014-01-01 01:00:00,0.0,11.22,0.0,8.174004,8.174004,0.0,0.0
2014-01-01 02:00:00,0.0,11.12,0.0,7.957390,7.957390,0.0,0.0
2014-01-01 03:00:00,0.0,10.67,0.0,7.029906,7.029906,0.0,0.0
2014-01-01 04:00:00,0.0,10.39,0.0,6.490870,6.490870,0.0,0.0


In [46]:
# Extracting Features needed for Random Forest later on
df['Hr'] = df.index.hour
df['Mon'] = df.index.month
df['Day'] = df.index.dayofweek

spline = SplineTransformer(n_knots=5, degree=3, extrapolation='periodic') # Using periodic for wrap around after 23:00 and 0:00
spline_feat = spline.fit_transform(df[['Hr']])
spline_cols = []

for i in range(spline_feat.shape[1]):
    col_name = f"spline_hr_{i + 1}"
    df[col_name] = spline_feat[:, i]
    spline_cols.append(col_name)

# 3hr trends to track sudden changes
df['Windspeed_mean_3h'] = df['Windspeed(m/s)'].rolling(window=3, min_periods=1).mean()
df['Windspeed_std_3h'] = df['Windspeed(m/s)'].rolling(window=3, min_periods=1).std().fillna(0)
df['GHI_mean_3h'] = df['GHI(W/m2)'].rolling(window=3, min_periods=1).mean()

df[['Hr'] + spline_cols + ['Windspeed_mean_3h' , 'Windspeed_std_3h' , 'GHI_mean_3h']].head(13).round(3)


,Hr,spline_hr_1,spline_hr_2,spline_hr_3,spline_hr_4,Windspeed_mean_3h,Windspeed_std_3h,GHI_mean_3h
Time,,,,,,,,
2014-01-01 00:00:00,0,0.167,0.667,0.167,0.000,11.370,0.000,0.000
2014-01-01 01:00:00,1,0.094,0.639,0.266,0.001,11.295,0.106,0.000
2014-01-01 02:00:00,2,0.046,0.567,0.380,0.007,11.237,0.126,0.000
2014-01-01 03:00:00,3,0.018,0.465,0.493,0.024,11.003,0.293,0.000
2014-01-01 04:00:00,4,0.005,0.351,0.588,0.056,10.727,0.368,0.000
2014-01-01 05:00:00,5,0.000,0.239,0.651,0.110,10.357,0.331,0.000
2014-01-01 06:00:00,6,0.000,0.146,0.665,0.189,10.043,0.331,0.000
2014-01-01 07:00:00,7,0.002,0.080,0.625,0.294,9.647,0.411,11.473
2014-01-01 08:00:00,8,0.010,0.038,0.544,0.409,8.637,1.459,86.390


In [49]:
# 1 hr lag features to understand previous state

df['Windspeed_lag_1hr'] = df['Windspeed(m/s)'].shift(1)
df['GHI_lag_1hr'] = df['GHI(W/m2)'].shift(1)

lags = ['Windspeed_lag_1hr' , 'GHI_lag_1hr']
df[lags] = df[lags].bfill() # backfill the first row with a valid value

df[['Windspeed(m/s)' , 'Windspeed_lag_1hr' , 'GHI(W/m2)' , 'GHI_lag_1hr']].head(10).round(3)

,Windspeed(m/s),Windspeed_lag_1hr,GHI(W/m2),GHI_lag_1hr
Time,,,,
2014-01-01 00:00:00,11.37,11.37,0.00,0.00
2014-01-01 01:00:00,11.22,11.37,0.00,0.00
2014-01-01 02:00:00,11.12,11.22,0.00,0.00
2014-01-01 03:00:00,10.67,11.12,0.00,0.00
2014-01-01 04:00:00,10.39,10.67,0.00,0.00
2014-01-01 05:00:00,10.01,10.39,0.00,0.00
2014-01-01 06:00:00,9.73,10.01,0.00,0.00
2014-01-01 07:00:00,9.20,9.73,34.42,0.00
2014-01-01 08:00:00,6.98,9.20,224.75,34.42
